In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('nothing.xlsx')

df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))
df['Discount_Number'] = df['Discount'].apply(extract_discount)

df.drop("Discount", axis=1, inplace=True)

brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai+', 'ai', 'aiplus', 'ai plus', 'a i'], 
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'redmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)


df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)


def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')


def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [2]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"Nothing Phone (4a) (White, 128 GB)",37999.0,40999.0,4.5,"5,025 Ratings & 492 Reviews",8,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 50MP + 8MP | 32MP Front Camera,5400,Snapdragon 7s Gen 4 Processor,https://www.flipkart.com/nothing-phone-4a-whit...,https://rukminim2.flixcart.com/image/312/312/x...,7,nothing,5025,492
1,"Nothing Phone (4a) (Pink, 256 GB)",43999.0,46999.0,4.5,954 Ratings & 81 Reviews,12,256,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 50MP + 8MP | 32MP Front Camera,5400,Snapdragon 7s Gen 4 Processor,https://www.flipkart.com/nothing-phone-4a-pink...,https://rukminim2.flixcart.com/image/312/312/x...,6,nothing,954,81
2,"Nothing Phone (3) (Black, 512 GB)",59999.0,94999.0,4.4,867 Ratings & 95 Reviews,16,512,16.94 cm (6.67 inch) Display,50MP + 50MP + 50MP | 50MP Front Camera,5500,8s Gen 4 Mobile Platform Processor,https://www.flipkart.com/nothing-phone-3-black...,https://rukminim2.flixcart.com/image/312/312/x...,36,nothing,867,95
3,"Nothing Phone (4a) Pro (Silver, 128 GB)",49999.0,NaN,4.5,"2,387 Ratings & 280 Reviews",8,128,17.35 cm (6.83 inch) Full HD+ AMOLED Display,50MP + 50MP + 8MP | 32MP Front Camera,5400,Snapdragon 7 Gen 4 Processor,https://www.flipkart.com/nothing-phone-4a-pro-...,https://rukminim2.flixcart.com/image/312/312/x...,0,nothing,2387,280
4,"Nothing Phone (4a) (Blue, 128 GB)",37999.0,40999.0,4.5,"5,025 Ratings & 492 Reviews",8,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 50MP + 8MP | 32MP Front Camera,5400,Snapdragon 7s Gen 4 Processor,https://www.flipkart.com/nothing-phone-4a-blue...,https://rukminim2.flixcart.com/image/312/312/x...,7,nothing,5025,492


In [3]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                9
Rating             0
Review text        0
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [4]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [5]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        0
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [6]:
df.to_excel('Cleaned_nothing.xlsx', index=False)